In [1]:
import geopandas as gpd

In [6]:
SWORD_directory = '/Volumes/PhD/SWORD/v17c/'

continent = 'SA'
dfG  = gpd.read_parquet(SWORD_directory + f'sword_{continent}_v17c_reaches.parquet')
dfN = gpd.read_parquet(SWORD_directory + f'sword_{continent}_v17c_nodes.parquet')


dfG = dfG.to_crs('EPSG:3857')
dfN = dfN.to_crs('EPSG:3857')

In [44]:
dfcheck  = gpd.read_file('/Volumes/PhD/SWORD/v17c/check/global_edges.gpkg')


In [46]:
dfcheckc = dfcheck[dfcheck['continent'] == 'sa']

In [121]:
mips = [
    680, 560, 1951, 2540, 1094, 2509, 381, 30, 35, 3247, 2957, 1033,
    1171, 1147, 113, 1788, 539, 2244, 2947, 236, 1378, 2443, 1617,
    659, 599, 2206, 2202, 1,
]
newMips = []
for m in mips:
    r = dfcheckc.loc[dfcheckc['main_path_id'] == m, 'reach_id'].to_list()
    newMips.append(int(dfG.loc[dfG['reach_id'] == r[0], 'main_path_id'].iloc[0]))

newMips[16] = 6000084 

# Tuning width & nr-channels

In [ ]:
import importlib
import linecache
import inspect

import pickle
from pathlib import Path

import incorporate_multichannel_segments as ims
import PELT
import PELT_tuning as pt
import PELT_finalize as pf

linecache.clearcache()

importlib.reload(ims)
importlib.reload(PELT)
importlib.reload(pt)
importlib.reload(pf)


In [ ]:
OUTDIR_TUNING = "PELT_outputs/01_width_channels/tuning"

grid_outputs = pt.PELT_grid_search(
    df=dfG,
    dfN=dfN,
    mips=newMips,
    input_windows=[0, 2, 3, 4, 5],
    PELT_penalties=(2.5, 5.0, 10.0, 20.0, 40.0, 80.0, 160.0),
    min_support_frac_runs_values=(0.05, 0.10, 0.15, 0.20, 0.25),
    stop_rel_improvement_values=(0.04, 0.045, 0.05, 0.06),
    pelt_feature_cols=("width_s", "nch_s"),
    consensus_cfg=PELT.DEFAULT_FROZEN_CONSENSUS_CONFIG,
    make_plots=False,
    print_timings=False,
    save_exports=True,
    show_progress=True,
    outdir=OUTDIR_TUNING,
    swot_node_dir="/Volumes/PhD/SWOT/RiverSP_D_parq/node/",
    swot_region="SA",
)


PELT grid search:   4%|▎         | 1/28 [10:16<4:37:38, 616.99s/reach, reach=6000212]

Component 0 is not calculated correctly


PELT grid search:  11%|█         | 3/28 [29:34<4:02:07, 581.12s/reach, reach=6000217]

Component 0 is not calculated correctly


PELT grid search:  25%|██▌       | 7/28 [49:14<1:59:20, 340.95s/reach, reach=6000007]

...

PELT grid search:  43%|████▎     | 12/28 [1:07:20<1:01:48, 231.78s/reach, reach=6000034]

Component 0 is not calculated correctly


PELT grid search:  57%|█████▋    | 16/28 [1:33:44<1:13:05, 365.43s/reach, reach=6000084]

Component 5 is not calculated correctly


/Users/6256481/Code/river-morphology-atlas/PELT.py:1003: RuntimeWarning: invalid value encountered in log
  width = np.where(width > 0, np.log(width), np.nan)
/Users/6256481/Code/river-morphology-atlas/PELT.py:1003: RuntimeWarning: invalid value encountered in log
  width = np.where(width > 0, np.log(width), np.nan)
/Users/6256481/Code/river-morphology-atlas/PELT.py:1003: RuntimeWarning: invalid value encountered in log
  width = np.where(width > 0, np.log(width), np.nan)
/Users/6256481/Code/river-morphology-atlas/PELT.py:1003: RuntimeWarning: invalid value encountered in log
  width = np.where(width > 0, np.log(width), np.nan)
/Users/6256481/Code/river-morphology-atlas/PELT.py:1003: RuntimeWarning: invalid value encountered in log
  width = np.where(width > 0, np.log(width), np.nan)
PELT grid search: 100%|██████████| 28/28 [2:37:51<00:00, 338.27s/reach, reach=6000323]  


In [167]:
grid_outputs = pt.calibrate_and_reapply_consensus_to_grid_outputs(
    grid_outputs=grid_outputs,
    save_exports=True,
    outdir=OUTDIR_TUNING,
    calibration_outdir=OUTDIR_TUNING,
    target_families=("w2", "w3", "w4", "w5"),
)


In [168]:
analysis_tables = pt.run_tuning_analysis(
    results_dict=grid_outputs["results_dict"],
    print_outputs=True,
)

with open(Path(OUTDIR_TUNING) / "PELT_analysis_tables.pkl", "wb") as f:
    pickle.dump(analysis_tables, f, protocol=pickle.HIGHEST_PROTOCOL)

final_inputs = pf.derive_finalization_inputs_from_analysis_tables(
    analysis_tables,
    consensus_cfg=grid_outputs["consensus_cfg"],
)

print("Winning family:", final_inputs["winning_family"])
print("Window key:", final_inputs["window_key"])
print("Final stable support count:", final_inputs["stable_support_count"])
print("Selector settings:")
for s in final_inputs["selector_settings"]:
    print(" ", s)



Family overview
  window_version  reaches  n_settings  n_pareto  mean_ch_all  \
0            raw       28          20         8  3533.931418   
1             w2       27          20        15  9637.833865   
2             w3       27          40        36  8702.646010   
3             w4       27          60        34  8012.525027   
4             w5       27          60        35  8499.405118   

   mean_centrality_all  mean_span_penalty_km_all  \
0             0.920747                  0.178688   
1             0.858954                  1.056259   
2             0.768814                  2.304153   
3             0.698509                  2.527320   
4             0.786607                  1.952027   

   mean_plateau_instability_all  rep_min_support_frac_runs  \
0                      9.436541                       0.20   
1                     12.500536                       0.15   
2                     28.640099                       0.15   
3                     42.324058      

# Tuning sinuosity & curvature

In [171]:
# Cell 1: imports + centerlines
import pickle
from pathlib import Path
import importlib

import PELT
import PELT_tuning as pt
import PELT_finalize as pf
import PELT_geometry_features as pgf
import PELT_segmentation_runner as psr

importlib.reload(PELT)
importlib.reload(pt)
importlib.reload(pf)
importlib.reload(pgf)
importlib.reload(psr)


<module 'PELT_segmentation_runner' from '/Users/6256481/Code/river-morphology-atlas/PELT_segmentation_runner.py'>

In [173]:

OUTDIR_TUNING = "PELT_outputs/02_sinuosity_curvature/tuning"

centerlines, centerline_qa = psr.build_centerlines_from_edges(
    dfG[dfG['main_path_id'].isin(newMips)],
    out_path=Path(OUTDIR_TUNING) / "centerlines.parquet",
    assert_all_linestring=True,
    endpoint_gap_tol=160,
    endpoint_gap_connected_tol=1e-6,
    graph_union_grid_size=1e-4,
)

geometry_feature_cfg = pgf.GeometryFeatureConfig(
    dist_col="dist_m",
    width_col="multi_width",
)

print(centerline_qa.head())
print("n centerlines:", len(centerlines))


100%|██████████| 28/28 [00:17<00:00,  1.64it/s]

   main_path_id geometry_type  is_linestring  is_multilinestring  is_empty  \
0       6000007    LineString           True               False     False   
1       6000009    LineString           True               False     False   
2       6000034    LineString           True               False     False   
3       6000041    LineString           True               False     False   
4       6000053    LineString           True               False     False   

       length_m  
0  1.297261e+06  
1  2.802397e+06  
2  1.231895e+06  
3  2.264216e+06  
4  2.822894e+06  
n centerlines: 28


In [174]:
# Cell 2: broad tuning run + consensus calibration
grid_outputs = pt.PELT_grid_search(
    df=dfG,
    dfN=dfN,
    mips=newMips,
    input_windows=[2, 3, 4, 5],  # no raw window for geometry features
    PELT_penalties=(2.5, 5.0, 10.0, 20.0, 40.0, 80.0, 160.0),
    min_support_frac_runs_values=(0.05, 0.10, 0.15, 0.20, 0.25),
    stop_rel_improvement_values=(0.04, 0.045, 0.05, 0.06),
    pelt_feature_cols=("sinu", "curv_int"),
    consensus_cfg=PELT.DEFAULT_FROZEN_CONSENSUS_CONFIG,
    centerlines=centerlines,
    centerline_id_col="main_path_id",
    centerline_geometry_col="line",
    node_geometry_col="geometry",
    geometry_feature_cfg=geometry_feature_cfg,
    check_centerline_orientation=True,
    make_plots=False,
    print_timings=False,
    save_exports=True,
    show_progress=True,
    outdir=OUTDIR_TUNING,
    swot_node_dir="/Volumes/PhD/SWOT/RiverSP_D_parq/node/",
    swot_region="SA",
)

grid_outputs = pt.calibrate_and_reapply_consensus_to_grid_outputs(
    grid_outputs=grid_outputs,
    save_exports=True,
    outdir=OUTDIR_TUNING,
    calibration_outdir=OUTDIR_TUNING,
    target_families=("w2", "w3", "w4", "w5"),
)


PELT grid search:   4%|▎         | 1/28 [17:27<7:51:35, 1047.98s/reach, reach=6000212]

Component 0 is not calculated correctly


PELT grid search:  11%|█         | 3/28 [1:01:08<8:42:53, 1254.93s/reach, reach=6000217]

Component 0 is not calculated correctly


PELT grid search:  21%|██▏       | 6/28 [1:41:22<6:25:10, 1050.50s/reach, reach=6000007]

...

PELT grid search:  43%|████▎     | 12/28 [2:01:06<1:29:27, 335.46s/reach, reach=6000034]

Component 0 is not calculated correctly


PELT grid search:  57%|█████▋    | 16/28 [2:32:09<1:31:55, 459.62s/reach, reach=6000084]

Component 5 is not calculated correctly


PELT grid search: 100%|██████████| 28/28 [7:49:04<00:00, 1005.15s/reach, reach=6000323]   


In [176]:
# Cell 3: tuning analysis + convert to finalization inputs
analysis_tables = pt.run_tuning_analysis(
    results_dict=grid_outputs["results_dict"],
    print_outputs=True,
)

with open(Path(OUTDIR_TUNING) / "PELT_analysis_tables.pkl", "wb") as f:
    pickle.dump(analysis_tables, f, protocol=pickle.HIGHEST_PROTOCOL)

final_inputs = pf.derive_finalization_inputs_from_analysis_tables(
    analysis_tables,
    consensus_cfg=grid_outputs["consensus_cfg"],
)

print("Winning family:", final_inputs["winning_family"])
print("Window key:", final_inputs["window_key"])
print("Final stable support count:", final_inputs["stable_support_count"])
print("Selector settings:")
for s in final_inputs["selector_settings"]:
    print(" ", s)



Family overview
  window_version  reaches  n_settings  n_pareto  mean_ch_all  \
0             w2       27          20        12  3914.267486   
1             w3       27          40        37  3340.488216   
2             w4       27          60        50  3079.544275   
3             w5       27          60        46  3070.671551   

   mean_centrality_all  mean_span_penalty_km_all  \
0             0.856533                  1.956510   
1             0.678177                  2.984779   
2             0.621383                  4.355001   
3             0.639945                  3.558630   

   mean_plateau_instability_all  rep_min_support_frac_runs  \
0                      7.461063                       0.10   
1                      5.714803                       0.10   
2                     10.949657                       0.25   
3                      6.780784                       0.10   

   rep_min_support_frac_windows_effective  rep_stop_rel_improvement  \
0                  